# 🌳 Visualização da Árvore de Decisão Minimax

Este notebook demonstra como a IA constrói e avalia a árvore de decisões para encontrar a jogada ideal usando o algoritmo Minimax.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'backend'))

from games.tic_tac_toe import TicTacToe
from engine.minimax_ai import MinimaxAI
print("Módulos importados com sucesso!")

## 1. Tabuleiro de Teste (Estado Quase Terminal)

Para visualizar a árvore de forma limpa, usaremos um tabuleiro onde restam poucos movimentos possíveis.

In [ ]:
game = TicTacToe()
# Tabuleiro com 7 posições ocupadas (apenas 2 posições livres)
state = {
    "board": [
        1, 2, 1,
        2, 1, 2,
        0, 0, 0
    ],
    "current_player": 1
}

def print_board(board):
    symbols = {0: '·', 1: 'X', 2: 'O'}
    for r in range(3):
        print(' ' + ' │ '.join(symbols[board[r*3 + c]] for c in range(3)))
        if r < 2:
            print(' ──┼───┼──')

print_board(state["board"])

## 2. Construindo a Árvore de Decisão Recursiva

Vamos estender a avaliação de nós e construir a estrutura hierárquica.

In [ ]:
def get_tree_nodes(state, depth, maximizing, last_move=None):
    result = game.check_result(state)
    score = game.evaluate(state)
    
    node = {
        "move": last_move,
        "board": list(state["board"]),
        "score": score,
        "over": result["over"],
        "winner": result["winner"],
        "children": []
    }
    
    if result["over"] or depth == 0:
        return node
        
    moves = game.get_valid_moves(state)
    for move in moves:
        next_state = game.apply_move(game.clone_state(state), move)
        child = get_tree_nodes(next_state, depth - 1, not maximizing, move)
        node["children"].append(child)
        
    # Minimax backpropagation score mapping
    if node["children"]:
        scores = [c["score"] for c in node["children"]]
        node["score"] = max(scores) if maximizing else min(scores)
        
    return node

tree = get_tree_nodes(state, depth=3, maximizing=True)
print("Árvore de decisão calculada com sucesso!")

## 3. Exibição Gráfica em Modo Texto

Exibe a árvore no formato de diretórios para facilitar a leitura das ramificações.

In [ ]:
def render_tree(node, indent="", is_last=True, is_root=False):
    connector = "" if is_root else ("└── " if is_last else "├── ")
    move_str = "Start" if node["move"] is None else f"Jogada {node['move']}"
    winner_str = "" if not node["over"] else (f" (Vencedor: {node['winner']})" if node["winner"] else " (Empate)")
    
    print(f"{indent}{connector}{move_str} [Score: {node['score']}]{winner_str} Board: {node['board']}")
    
    new_indent = indent + ("    " if is_last or is_root else "│   ")
    num_children = len(node["children"])
    for i, child in enumerate(node["children"]):
        render_tree(child, new_indent, is_last=(i == num_children - 1))

render_tree(tree, is_root=True)